# Pré-processamento 9 — Revisão de compatibilidade entre palavra e categoria
## Decisões aprovadas
Aplicar 10 correções do grupo Verbo Principal Gerúndio. Não alterar LEMA nem grafia.
O código aceita categorias já corrigidas, permitindo reexecução. Valida palavras pela identidade e pelo lema, não pela antiga posição da linha.
Para acrescentar remoções, edite a lista e reexecute desde o carregamento.
O salvamento mostra as contagens da execução e o total final da base. Arquivos intermediários são criados em pasta temporária e limpos automaticamente ao concluir ou ocorrer uma exceção. As versões anteriores são mantidas pelo Git.


In [1]:
from pathlib import Path
import json, hashlib, os, subprocess
import pandas as pd
from IPython.display import display
arquivo = Path("Corpus_SANTOS_TONI_FDC_tratado_sem_repeticoes.xlsx")
base = pd.read_excel(arquivo)
base_original = base.copy(deep=True)
hash_entrada = hashlib.sha256(arquivo.read_bytes()).hexdigest()
print("Registros carregados:", len(base))


Registros carregados: 7776


## 1. Função de correção de categoria e preparação do resultado

In [2]:
# Função compartilhada pelos grupos; todo o código de correção está neste notebook.
# Altera exclusivamente CATMORF, sem atribuir valores em LEMA.
def aplicar_correcoes_categoria(resultado, correcoes):
    antes = resultado.copy(deep=True)
    aplicadas = []
    for item in correcoes:
        mascara = resultado["PALAVRA"].eq(item["PALAVRA"])
        assert mascara.sum() == 1, f"Palavra ausente ou duplicada: {item['PALAVRA']}"
        indice = resultado.index[mascara][0]
        atual = resultado.at[indice, "CATMORF"]
        assert atual in (item["CATMORF"], item["CATEGORIA_SUGERIDA"]), f"Categoria mudou: {item['PALAVRA']}"
        assert resultado.at[indice, "LEMA"] == item["LEMA"], f"Lema mudou: {item['PALAVRA']}"
        if atual != item["CATEGORIA_SUGERIDA"]:
            resultado.at[indice, "CATMORF"] = item["CATEGORIA_SUGERIDA"]
            aplicadas.append({**item, "CATEGORIA_ANTERIOR": atual})
    pd.testing.assert_frame_equal(antes.drop(columns="CATMORF"), resultado.drop(columns="CATMORF"))
    return aplicadas

resultado = base.copy(deep=True)
alteracoes = []
correcoes_aprovadas = []


## 1.1. Verbo Principal Gerúndio — aplicar 10 correções aprovadas

In [3]:
# Lista completa das decisões aprovadas — Verbo Principal Gerúndio
correcoes_gerundio = [{'LINHA_EXCEL': 233,
  'PALAVRA': 'ajeitei',
  'LEMA': 'ajeitar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Primeira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito perfeito na primeira pessoa do singular, respectivamente de ajeitar '
                   'e amassar; não é gerúndio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 239,
  'PALAVRA': 'ajudar',
  'LEMA': 'ajudar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Forma de infinitivo, não de gerúndio. Preservar o lema cadastrado, mesmo '
                   'quando não corresponde à palavra.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 296,
  'PALAVRA': 'almoçou',
  'LEMA': 'almoçar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Terceira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito perfeito de almoçar, terceira pessoa do singular; o gerúndio é '
                   'almoçando.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 322,
  'PALAVRA': 'amadurecer',
  'LEMA': 'amaciar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo',
  'JUSTIFICATIVA': 'Forma de infinitivo, não de gerúndio. Preservar o lema cadastrado, mesmo '
                   'quando não corresponde à palavra.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 346,
  'PALAVRA': 'amassei',
  'LEMA': 'amassar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Indicativo Passado Primeira Pessoa Singular',
  'JUSTIFICATIVA': 'Pretérito perfeito na primeira pessoa do singular, respectivamente de ajeitar '
                   'e amassar; não é gerúndio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 382,
  'PALAVRA': 'andarem',
  'LEMA': 'andar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo Terceira Pessoa Plural',
  'JUSTIFICATIVA': 'Admite infinitivo pessoal na terceira pessoa do plural ou futuro do subjuntivo '
                   'na mesma pessoa. A proposta de infinitivo é uma opção válida, sem preferência '
                   'obrigatória; não é gerúndio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 383,
  'PALAVRA': 'andares',
  'LEMA': 'andar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Verbo Principal Infinitivo Segunda Pessoa Singular',
  'JUSTIFICATIVA': 'Admite infinitivo pessoal ou futuro do subjuntivo na segunda pessoa do '
                   'singular; também pode ser plural do substantivo andar. A proposta verbal é uma '
                   'opção válida, sem preferência obrigatória; não é gerúndio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 426,
  'PALAVRA': 'ao',
  'LEMA': 'anunciar',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Preposição + Artigo Definido Masculino Singular',
  'JUSTIFICATIVA': 'Contração de a com o, compatível com o rótulo já existente na base. Não é '
                   'gerúndio de anunciar indicado no lema.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 902,
  'PALAVRA': 'bem-vindo',
  'LEMA': 'bem-vindo',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Adjetivo Qualificativo Masculino Singular',
  'JUSTIFICATIVA': 'Adjetivo masculino singular; também pode ser usado em expressão de saudação. A '
                   'terminação vindo não torna o composto um gerúndio.',
  'DECISAO': 'Aprovada'},
 {'LINHA_EXCEL': 4676,
  'PALAVRA': 'marimbondo',
  'LEMA': 'marimbondo',
  'CATMORF': 'Verbo Principal Gerúndio',
  'CATEGORIA_SUGERIDA': 'Nome Comun Masculino Singular',
  'JUSTIFICATIVA': 'Substantivo masculino singular que designa o inseto; terminar em ndo não '
                   'caracteriza gerúndio por si só.',
  'DECISAO': 'Aprovada'}]
# Aplicação efetiva na base em memória:
alteracoes_grupo = aplicar_correcoes_categoria(resultado, correcoes_gerundio)
alteracoes.extend(alteracoes_grupo)
correcoes_aprovadas.extend(correcoes_gerundio)
print("Grupo: Verbo Principal Gerúndio")
print("Correções aprovadas:", len(correcoes_gerundio))
print("Aplicadas nesta execução:", len(alteracoes_grupo))
print("Já estavam corrigidas:", len(correcoes_gerundio) - len(alteracoes_grupo))
display(pd.DataFrame(correcoes_gerundio)[["PALAVRA", "LEMA", "CATMORF", "CATEGORIA_SUGERIDA", "DECISAO"]])


Grupo: Verbo Principal Gerúndio
Correções aprovadas: 10
Aplicadas nesta execução: 0
Já estavam corrigidas: 10


,PALAVRA,LEMA,CATMORF,CATEGORIA_SUGERIDA,DECISAO
0,ajeitei,ajeitar,Verbo Principal Gerúndio,Verbo Principal Indicativo Passado Primeira Pe...,Aprovada
1,ajudar,ajudar,Verbo Principal Gerúndio,Verbo Principal Infinitivo,Aprovada
2,almoçou,almoçar,Verbo Principal Gerúndio,Verbo Principal Indicativo Passado Terceira Pe...,Aprovada
3,amadurecer,amaciar,Verbo Principal Gerúndio,Verbo Principal Infinitivo,Aprovada
4,amassei,amassar,Verbo Principal Gerúndio,Verbo Principal Indicativo Passado Primeira Pe...,Aprovada
5,andarem,andar,Verbo Principal Gerúndio,Verbo Principal Infinitivo Terceira Pessoa Plural,Aprovada
6,andares,andar,Verbo Principal Gerúndio,Verbo Principal Infinitivo Segunda Pessoa Sing...,Aprovada
7,ao,anunciar,Verbo Principal Gerúndio,Preposição + Artigo Definido Masculino Singular,Aprovada
8,bem-vindo,bem-vindo,Verbo Principal Gerúndio,Adjetivo Qualificativo Masculino Singular,Aprovada
9,marimbondo,marimbondo,Verbo Principal Gerúndio,Nome Comun Masculino Singular,Aprovada


## 2. Palavras identificadas como fora do escopo

In [4]:
palavras_remover = []

total_antes = len(resultado)
mascara_remocao = resultado["PALAVRA"].isin(palavras_remover)
removidas = resultado.loc[mascara_remocao].copy()
nao_encontradas = sorted(set(palavras_remover) - set(removidas["PALAVRA"]))
resultado = resultado.loc[~mascara_remocao].reset_index(drop=True)
quantidade_removida = len(removidas)
assert total_antes - quantidade_removida == len(resultado)
print("Remoções previstas nesta execução:", quantidade_removida)
print("Total previsto na base:", len(resultado))
if nao_encontradas:
    print("Palavras já removidas ou não encontradas:", nao_encontradas)
display(removidas[["PALAVRA", "LEMA", "CATMORF"]])


Remoções previstas nesta execução: 0
Total previsto na base: 7776


,PALAVRA,LEMA,CATMORF


## 3. Salvamento e contagens

In [5]:
assert hashlib.sha256(arquivo.read_bytes()).hexdigest() == hash_entrada, "A fonte mudou durante a execução."
retidas = base_original.loc[~base_original["PALAVRA"].isin(palavras_remover)].reset_index(drop=True)
pd.testing.assert_frame_equal(retidas.drop(columns="CATMORF"), resultado.drop(columns="CATMORF"))
from tempfile import TemporaryDirectory
# A limpeza inclui arquivos de inspeção gerados pela ferramenta, mesmo em caso de erro.
with TemporaryDirectory(prefix="temporarios_", dir=arquivo.resolve().parent) as pasta_temporaria:
    temporario = Path(pasta_temporaria) / "base_verificacao.xlsx"
    pacote = Path(pasta_temporaria) / "operacao.json"
    matriz = [list(resultado.columns)] + json.loads(resultado.to_json(orient="values", force_ascii=False))
    pacote.write_text(json.dumps({
        "source": str(arquivo.resolve()), "target": str(temporario.resolve()),
        "originalRows": len(base_original) + 1, "values": matriz
    }, ensure_ascii=False), encoding="utf-8")
    runtime = Path(os.environ.get("USERPROFILE", str(Path.home()))) / ".cache/codex-runtimes/codex-primary-runtime/dependencies/node"
    node = runtime / "bin/node.exe"
    ambiente = os.environ.copy()
    ambiente["CODEX_NODE_PACKAGES"] = str(runtime / "node_modules")
    codigo_js = "\nimport fs from 'node:fs/promises';\nimport {createRequire} from 'node:module';\nimport {pathToFileURL} from 'node:url';\nconst require = createRequire(process.env.CODEX_NODE_PACKAGES + '/../package.json');\nconst {FileBlob, SpreadsheetFile} = await import(pathToFileURL(require.resolve('@oai/artifact-tool')).href);\nconst job=JSON.parse(await fs.readFile(process.argv[1],'utf8'));\nconst wb=await SpreadsheetFile.importXlsx(await FileBlob.load(job.source));\nconst sheet=wb.worksheets.getItem('Sheet1');\nsheet.getRange('A1:H'+job.originalRows).clear({applyTo:'contents'});\nsheet.getRange('A1:H'+job.values.length).values=job.values;\nwb.recalculate();\nconsole.log((await wb.inspect({kind:'region',sheetId:'Sheet1',range:'A1:C5',maxChars:1000})).ndjson);\nconst out=await SpreadsheetFile.exportXlsx(wb);\nawait out.save(job.target);\n"
    processo = subprocess.run([str(node), "--input-type=module", "-e", codigo_js, str(pacote.resolve())],
                             env=ambiente, capture_output=True, text=True, encoding="utf-8")
    if processo.returncode:
        raise RuntimeError(processo.stderr or processo.stdout)
    verificada = pd.read_excel(temporario)
    pd.testing.assert_frame_equal(resultado, verificada, check_dtype=False)
    assert hashlib.sha256(arquivo.read_bytes()).hexdigest() == hash_entrada
    if not resultado.equals(base_original):
        os.replace(temporario, arquivo)
print("Correções aplicadas nesta execução:", len(alteracoes))
print("Remoções nesta execução:", quantidade_removida)
print("Total atual na base:", len(verificada))
print("Lemas e demais colunas das linhas mantidas: preservados.")


Correções aplicadas nesta execução: 0
Remoções nesta execução: 0
Total atual na base: 7776
Lemas e demais colunas das linhas mantidas: preservados.
